In [11]:
import roboticstoolbox as rtb
import spatialmath as sm
import numpy as np


# Ejercicio 1

Dado el robot SCARA con la asignación de ternas vista en clase y los parámetros:

- a1=0.2
- a2=0.3

Calcular la orientación del eslabón 1 según la terna 2 expresado como un ángulo de rotación sobre z2, cuando se tienen las variables articulares:

q = [pi/3 -pi/6 100 -0.1*pi]'

(Unidades: radianes. Precisión: 0.01)

In [25]:
# Modelo SCARA mediante parámetros Denavit-Hartenberg
scara = rtb.DHRobot([
    rtb.RevoluteDH(a=0.2, qlim=[-np.pi, np.pi]),        # q1: rotación plana
    rtb.RevoluteDH(a=0.3, qlim=[-np.pi, np.pi]),        # q2: rotación plana
    rtb.PrismaticDH(qlim=[0.0, 0.5]),                   # q3: desplazamiento vertical 
    rtb.RevoluteDH(qlim=[-np.pi, np.pi])                # q4: orientación final
])

print(scara)

q = np.r_[np.pi/3, -np.pi/6, 0.1, -0.1*np.pi]

A = scara.fkine_all(q)
R21 = A[1].R.T @ A[2].R

print(R21.T)

theta,k = sm.SO3(R21.T).angvec()
print(theta*k[2])



DHRobot: , 4 joints (RRPR), dynamics, standard DH parameters
┌──────┬─────┬─────┬──────┬─────────┬────────┐
│  θⱼ  │ dⱼ  │ aⱼ  │  ⍺ⱼ  │   q⁻    │   q⁺   │
├──────┼─────┼─────┼──────┼─────────┼────────┤
│  q1  │   0 │ 0.2 │ 0.0° │ -180.0° │ 180.0° │
│  q2  │   0 │ 0.3 │ 0.0° │ -180.0° │ 180.0° │
│ 0.0° │  q3 │   0 │ 0.0° │     0.0 │    0.5 │
│  q4  │   0 │   0 │ 0.0° │ -180.0° │ 180.0° │
└──────┴─────┴─────┴──────┴─────────┴────────┘

┌──┬──┐
└──┴──┘

[[ 0.8660254 -0.5        0.       ]
 [ 0.5        0.8660254  0.       ]
 [ 0.         0.         1.       ]]
0.5235987755982988


# Ejercicio 2
Para el robot IRB140, con la asignación de ternas vistas en clase, calcular la posición alcanzada en y0 con la muñeca, cuando las variables articulares toman el valor:

q=[0 0 0 0 pi/2 0]

Unidades: mm

Precisión:1

In [16]:
irb140 = rtb.DHRobot(
    [
        rtb.RevoluteDH(alpha=-np.pi/2,a=70),
        rtb.RevoluteDH(a=360),
        rtb.RevoluteDH(alpha=np.pi/2),
        rtb.RevoluteDH(d=380, alpha=-np.pi/2),
        rtb.RevoluteDH(alpha=np.pi/2),
        rtb.RevoluteDH()
    ], name="IRB140")

print(irb140)

q = np.r_[0, 0, 0, 0, np.pi/2, 0]

A = irb140.fkine(q)
print(A.t[1])

DHRobot: IRB140, 6 joints (RRRRRR), dynamics, standard DH parameters
┌─────┬─────┬─────┬────────┐
│ θⱼ  │ dⱼ  │ aⱼ  │   ⍺ⱼ   │
├─────┼─────┼─────┼────────┤
│  q1 │   0 │  70 │ -90.0° │
│  q2 │   0 │ 360 │   0.0° │
│  q3 │   0 │   0 │  90.0° │
│  q4 │ 380 │   0 │ -90.0° │
│  q5 │   0 │   0 │  90.0° │
│  q6 │   0 │   0 │   0.0° │
└─────┴─────┴─────┴────────┘

┌──┬──┐
└──┴──┘

0.0


# Ejercicio 3

Para el IRB140 con la asignación de ternas vistas en clase, se desea  realizar un movimiento que lleva la terna 6 a la pose descripta por los vectores:

- posicion = [0,0,650]'

- rotacion = [1,0,0,0]'

- configuracion =[1 -1 1]

Indicar cuáles de las siguientes afirmaciones son correctas.

(ojo: la rotación está expresada como un cuaternión)


Seleccione una o más de una:

a. El punto no puede ser alcanzado con la configuración de brazo pedida.

b. Existe una única solución pues se ha indicado el vector de configuraciones

c. El punto deseado no puede alcanzarse por estar fuera de rango

d. Existen infinitas combinaciones de q1 y q6 que son solución para alcanzar esta pose

In [20]:
pose = sm.SE3.Trans(0,0,650)
print(pose)

sol = irb140.ikine_LM(pose)
print(sol)

print(irb140.fkine(sol.q))

   1         0         0         0         
   0         1         0         0         
   0         0         1         650       
   0         0         0         1         

IKSolution: q=[2.622, -1.176, 0.5952, -3.142, -0.5807, 0.5193], success=True, iterations=10, searches=1, residual=8.18e-07
   1         2.913e-07 -2.217e-07  8.811e-13  
  -2.913e-07  1        -3.876e-07 -5.116e-13  
   2.217e-07  3.876e-07  1         650       
   0         0         0         1         



# Ejercicio 4

Para el robot IRB140 con la asignación de ternas vista en clase, calcular el error en valor absoluto en la dirección x0 al querer alcanzar la

- POSE=[1 0 0 500;0 1 0 100;0 0 1 400;0 0 0 1]
- conf=[1 -1 1]

cuando d4 tiene en realidad un valor 380.1 mientras que el software utiliza el valor nominal para el cálculo

Precisión: 0.0001

Unidades: mm

In [ ]:
POSE = sm.SE3.Trans(500,100,400)
print(POSE)

irb140.ikine_LM(POSE)

   1         0         0         500       
   0         1         0         100       
   0         0         1         400       
   0         0         0         1         

